# Anchor Refinery
Created by Quillan Shimp with the use of Claude. Images from Legacy Survey.  

### How to Use
Start the vetting program by running all cells.  
Input your username.  
Use the morphology dropdown to change galaxy type, use the style buttons to change image type.  
Select any number of galaxies. Then assign an alternate morphology, bad anchor attribute, and/or write notes.  
Click the galaxy again while the border is blue to deselect.  
You may change the number of galaxies per page and number of columns in the Settings cell.  
You may click save to save your progress and exit to end your session.  
### Advice
When initiated, the galaxies will not display correctly. Press any button that changes the display and it will function normally.  
Click save intermittently while vetting (I do so every hundred galaxies), you never know what will happen.  
Write notes before assigning morphology. Once the type is assigned, the galaxy is uninteractable.  
If memory overloads during a session and the page refreshes, don't panic. The tool will reload where you left off and you can save from there.  
Avoid overusing the bad anchor option. Removing the anchor won't remove the galaxy from the classifier! Whether some galaxies should not be classified should be discussed.
If you think of any improvement, let me know!
### The Sample
The first 450 galaxies or so of each type are from Julia's original anchor set. They were vetted with all image settings available. The remaining galaxies used John's SGA-2025 group-centered cutouts (SSL mode). These galaxies have only been vetted once. The final 400 or so irregulars were notably vetted within a 5 hour timespan and likely need careful attention to remove spirals. I avoided classifying grouped galaxies whenever ambiguous, but some galaxy groups survived with potentially unexpected primary galaxies leading to laughably wrong classification.

### 1.
Julia's anchors available in SGA 2025 are generated using Select_Galaxies.ipynb

In [1]:
import numpy as np
import pandas as pd

import glob
import h5py
import os

import matplotlib.pyplot as plt
import matplotlib.image as mpimg # displaying images

from SGA.qa import sdss_rgb # displaying SGA-2025 images

from IPython.display import display, clear_output # for display functions
import ipywidgets as widgets # for buttons

from cutout_vetter import CutoutVetter, ReviewerLogin, confusion_matrix_report # interactive image grid

In [2]:
# Taken from /global/homes/q/qshimp/SGA/doc/tutorials/SGA-ssl.ipynb
def build_cutout_index(ssl_dir):
    """
    Return {(region, sgaid): (hdf5_path, row_index)}
    for fast image retrieval.
    """
    files = sorted(glob.glob(os.path.join(ssl_dir, "ssl-cutouts-dr11-*.hdf5")))
    if not files:
        raise FileNotFoundError(f"No cutout files found in {ssl_dir}")

    index = {}
    for f in files:
        filename = os.path.basename(f)
        
        if "dr11-south" in filename:
            region = "dr11-south"
        elif "dr11-north" in filename:
            region = "dr11-north"
        else:
            raise ValueError(f"Could not determine region from filename: {filename}")
            
        with h5py.File(f, "r") as H:
            sgaids = H["sgaid"][:]
            
            for i, sgaid in enumerate(sgaids):
                key = (region, int(sgaid))
                if key in index:
                    print(f"WARNING: duplicate key found: {key}")
                index[key] = (f, i)

    print(f"Total unique (region, SGAID) pairs indexed: {len(index):,}")

    return index

# Load legacy survey jpgs
def load_jpg(row):
    sgaid = row["SGAID"]
    base = "/pscratch/sd/q/qshimp/Cutouts/sga2025/Anchor_jpgs"
    matchesM = glob.glob(f"{base}/Model/{sgaid}_*.jpg")
    matchesR = glob.glob(f"{base}/Residual/{sgaid}_*.jpg")
    matchesI = glob.glob(f"{base}/Image/{sgaid}_*.jpg")

    model = np.flipud(mpimg.imread(matchesM[0])) if matchesM else None
    residual = np.flipud(mpimg.imread(matchesR[0])) if matchesR else None
    image = np.flipud(mpimg.imread(matchesI[0])) if matchesI else None

    return model, residual, image

def lookup_by_morph(morph_label, df):
    return df[df["Morphology"] == morph_label].copy()

In [3]:
SSL_DIR = '/global/cfs/cdirs/desicollab/users/ioannis/SGA/2025/ssl'
CATALOG = '/global/cfs/cdirs/desicollab/users/qshimp/anchors/catalog_4400.csv'
anchor_catalog = pd.read_csv(CATALOG)
cutout_index = build_cutout_index(SSL_DIR)

Total unique (region, SGAID) pairs indexed: 445,693


In [4]:
# Settings
N_COLS = 10
N_PER_PAGE = 50

In [5]:
%matplotlib widget

login = ReviewerLogin(
    morph_options=sorted(anchor_catalog["Morphology"].unique()),
    data_lookup_fn=lookup_by_morph,
    df=anchor_catalog,
    cutout_index=cutout_index,
    sdss_rgb_fn=sdss_rgb,
    load_jpg_fn=load_jpg,
    ncols=N_COLS,
    n_per_page=N_PER_PAGE,
    figsize_per=2
)

In [8]:
matrix = confusion_matrix_report(
    save_dir="/global/cfs/cdirs/desicollab/users/qshimp/anchors",
    morph_options=sorted(anchor_catalog["Morphology"].unique()),
    username="qshimp"
)
matrix

,Elliptical,Irregular,Lenticular,Spiral,Bad anchor
Elliptical,0,0,0,0,0
Irregular,0,0,0,0,0
Lenticular,0,0,0,0,0
Spiral,0,0,0,0,0


# To do:
1. Allow page selection 
2. Create user system
3. Create exit button next to save button that returns to input username screen
5. Create getInfo function to enable users to check progress (matrix style display)
6. Change save from separate csvs to one master file (for each user)

## Other options:
1. Confirmed galaxies disappear instead of red border
2. Quality of life like "Loading..." or "Timed out!" to reduce waiting